In [4]:
!pip install evaluate
!pip install bert_score

In [5]:
import pandas as pd
from evaluate import load
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util

In [20]:
data = pd.read_csv('sarcasm.txt', sep='\t')
data = data.sample(frac=1, random_state=42, ignore_index=True)
data

,Style 1,Style 2
0,Who's ready for monsoon storms allllll weekend...,There are going to be monsoon storms all weeke...
1,Idk how some girls put on makeup everyday. Is ...,I don't know why girls put on makeup everyday.
2,Turnover is the best band to cry to/drive to/d...,"I would say, turnover is the best band to do a..."
3,It's nice to see some heritage organisations a...,The National Trust don't care about their staf...
4,Someone in the pub just described the England ...,Someone in the pub just described the England ...
...,...,...
688,"Let me guess, it has to be Charlotte Tilbury m...",She just wants you to use her make-up and skin...
689,it’s always the girls I don’t even know that w...,I don’t like people who talk bad about me
690,Oh? You're a real anime fan? Name all the Boku...,Real fans don't have to prove themselves to an...
691,just met the love of my life don't text,just met this man i have been following for tw...


# Задача 1: Train / Val / Test поделба

- правиме поделба бидејќи ако истите реченици се и пример и тестови, моделот ќе го види точниот одговор во prompt-от и ќе го препише.
  - **train_data** = земаме првите 80%
  - **val_data** = земаме од 80% до 90%
  - **test_data** = ги земаме последните 10%

In [21]:
train_size = int(0.8 * len(data))
val_size = int(0.1 * len(data))

train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

In [22]:
len(train_data), len(val_data), len(test_data)

(554, 69, 70)

In [23]:
source_column = 'Style 1' #sarcastic sentences
target_column = 'Style 2' #sincere sentences

# Задача 1.2: Претворање на колоните во листи (поради tokenizer и embedding)

In [24]:
train_source = train_data[source_column].values.tolist()
train_target = train_data[target_column].values.tolist()

In [25]:
val_source = val_data[source_column].values.tolist()
val_target = val_data[target_column].values.tolist()

In [26]:
test_source = test_data[source_column].values.tolist()
test_target = test_data[target_column].values.tolist()

# Задача 2: Embeddings од train, query embeddings од test
- Целта е за секој тест-tweet да се најдат најсличните tweet-ови од train, па нивните парови да станат примери во prompt-от. За да се споредуваат tweet-ови според значење, прво секој се претвора во embedding (листа од 768 броеви). **Сличните tweet-ови имаат слични броеви**.
- Две улоги:
  - **корпус** (train_emb) е каде се бара
  - **queries** (val_emb, test_emb) е за што се бара, односно tweet-ови за кои ни требаат примери

In [27]:
embedding_model = SentenceTransformer('all-distilroberta-v1')

train_emb = embedding_model.encode(train_source, batch_size=64)
val_emb = embedding_model.encode(val_source, batch_size=64)
test_emb = embedding_model.encode(test_source, batch_size=64)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Задача 3: Semantic search, top_K најслични
- секој tweet има embedding. Semantic search за секој val/test tweet ги споредува неговите броеви со броевите на сите train tweets и враќа 5 најслични. Тие ќе бидат примерите во prompt-от.
- Споредбата е со косинусна сличност, бројка од 0 до 1. Колку е поблиску до 1, толку е послично.

In [28]:
val_hits = util.semantic_search(val_emb, train_emb, top_k=5)
test_hits = util.semantic_search(test_emb, train_emb, top_k=5)
test_hits[0]

[{'corpus_id': 65, 'score': 0.37111377716064453},
 {'corpus_id': 84, 'score': 0.37100493907928467},
 {'corpus_id': 77, 'score': 0.36699774861335754},
 {'corpus_id': 227, 'score': 0.365402489900589},
 {'corpus_id': 117, 'score': 0.34541207551956177}]

# Задача 4: BERTScore на најсличните

In [33]:
bertscore = load('bertscore')

retrieved = []
for hits in test_hits:
  i = hits[0]['corpus_id']
  retrieved.append(train_target[i])

result = bertscore.compute(predictions=retrieved, references=test_target, model_type='distilroberta-base')
retrieval_f1 = sum(result['f1']) / len(result['f1'])
retrieval_f1

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.8667723962238857

# Задача 5: Prompt со примери и модел

In [35]:
model_name = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [36]:
def build_prompt(sentence, hits, k):
    examples = ''
    for hit in hits[:k]:
        i = hit['corpus_id']
        examples += f'Input: {train_source[i]} Output: {train_target[i]}\n'
    return f'Rewrite the sarcastic text as a sincere, non-sarcastic text.\nExamples:\n{examples}Input: {sentence} Output: '

In [38]:
def translate(prompt):
    tokens = tokenizer([prompt], return_tensors='pt')
    out = model.generate(**tokens, max_length=80)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompt = build_prompt(test_source[0], test_hits[0], 3)
print(prompt)
print('Model:', translate(prompt))
print('Reference:', test_target[0])

Rewrite the sarcastic text as a sincere, non-sarcastic text.
Examples:
Input: Going to M&amp;S food will be the highlight of my day later, can’t wait👌🏻😅 Output: “I love M&S food, can’t wait to buy dinner there later.”
Input: Just getting some WD40 on my letter box to help the poor postman with his delivery today #valentinesday2021 #WD40 Output: Fed up of seeing forced romance, don't be told when to do something nice for a person you love.
Input: ima get up early tomorrow and make some tea and sit on the patio oh how I love to imagine things im never gonna do Output: I would say, "I wish I could get up early and enjoy some tea on the patio."
Input: Enjoy the bars and vineyards✌🏻
Make sure to send postcards to the people dying in your local hospitals.  I’m sure they’ll be glad you’re not bored anymore Output: 
Model: Enjoy the bars and vineyards. Send postcards to the people dying in your local hospitals. I’m sure they’ll be glad you’re not bored anymore.
Reference: If you’re going to ba

In [39]:
def generate_and_score(sources, targets, hits_list, k):
    predictions = []
    for sentence, hits in zip(sources, hits_list):
        predictions.append(translate(build_prompt(sentence, hits, k)))
    result = bertscore.compute(predictions=predictions, references=targets, model_type='distilroberta-base')
    return sum(result['f1']) / len(result['f1']), predictions

In [ ]:
val_scores = {}
for k in [1, 3, 5]:
    val_scores[k], _ = generate_and_score(val_source, val_target, val_hits, k)
    print(f'K={k}: {val_scores[k]:.4f}')

best_k = max(val_scores, key=val_scores.get)
best_k

K=1: 0.9047
K=3: 0.9050


In [43]:
model_f1, predictions = generate_and_score(test_source, test_target, test_hits, best_k)

print('Most similar train sentence:', round(retrieval_f1, 4))
print('Model with examples:', round(model_f1, 4))

Most similar train sentence: 0.8668
Model with examples: 0.8962


In [44]:
for i in range(5):
    print('Input:    ', test_source[i])
    print('Model:    ', predictions[i])
    print('Reference:', test_target[i])
    print()

Input:     Enjoy the bars and vineyards✌🏻
Make sure to send postcards to the people dying in your local hospitals.  I’m sure they’ll be glad you’re not bored anymore
Model:     Enjoy the bars and vineyards. Send postcards to the people dying in your local hospitals. I’m sure they’ll be glad you’re not bored anymore.
Reference: If you’re going to bars and vineyards right now, just keep in mind the people who are dying in hospitals from COVID. Your entertainment is not worth their lives. 

Input:     my computer does this fun thing now where it shakes in a way that exactly replicates what happens to my vision when i forget my meds so that’s exciting !
Model:     my computer does this fun thing now where it shakes in a way that exactly replicates what happens to my vision when I forget my meds so that’s exciting !
Reference: Yes

Input:     i love jumping on zoom meetings where everyone is talking about their kids &amp; i'm stuffing my face with taco bell
Model:     I love jumping on zoom